# Week 8 · Topic 1 — Calling a Web API

**Box we are building: INPUT, from outside**

Everything you have loaded so far has been a file you already had. This week
it is a real service, somewhere else, that you ask for data.

> **Already know this?** Skip to the *Topic 1 Challenge*.

---
## 1. A GET request asks a URL for something

In [1]:
import requests

response = requests.get("https://api.github.com/orgs/python")
print(response.status_code)

200


```
your code                the API
    │                        │
    │──── GET request ──────▶│
    │                        │  looks it up
    │◀─── response ──────────│
    │   (status code + JSON)
```

### Your turn
GET `https://api.github.com/repos/python/cpython` and print its status code.

In [2]:
# your code here
import requests
response = requests.get("https://api.github.com/repos/python/cpython")
print(response.status_code)

200


---
## 2. Status codes tell you what happened

200s = OK. 400s = you got something wrong. 500s = their fault, not yours.

In [3]:
print(requests.get("https://api.github.com/orgs/python").status_code)              # 200
print(requests.get("https://api.github.com/this-does-not-exist-xyz").status_code)     # 404

200
404


### Your turn
GET a URL you know does not exist and print its status code.

In [4]:
# your code here
print(requests.get("https://api.github.com/yun"))

<Response [404]>


---
## 3. What is JSON, and what does `.json()` do?

**JSON** (JavaScript Object Notation) is the plain-text format that almost every API uses to send data. It is only text, but it is written so that it looks like Python dictionaries and lists:

| JSON text | Python, after `.json()` |
|---|---|
| `{"name": "Rex", "age": 4}` (an *object*) | a dictionary |
| `["dog", "cat"]` (an *array*) | a list |
| `"Rex"` | a string |
| `4` or `4.5` | an int or a float |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

A few rules: text is always in **double** quotes, keys are always text, and there are no trailing commas. Values can be nested: an object inside an object, or a list of objects. A list of objects is exactly what a table of records looks like.

The reply from `requests` is one long piece of **text**. `response.json()` turns that text into real Python dictionaries and lists, so you can use `data["login"]` or loop over the records.

In [3]:
import json

text = '{"name": "Rex", "age": 4, "vaccinated": true, "owner": null, "tags": ["dog", "friendly"]}'
print(type(text))                    # just text

pet = json.loads(text)               # this is what response.json() does for you
print(type(pet))                     # a dictionary
print(pet["name"], pet["vaccinated"], pet["owner"], pet["tags"][0])

<class 'str'>
<class 'dict'>
Rex True None dog


The same thing with a real API:

In [4]:
response = requests.get("https://api.github.com/orgs/python")
data = response.json()
print(data["login"], data["public_repos"])

python 99


### Your turn
GET `https://api.github.com/repos/python/cpython` and print its `"stargazers_count"`.

In [6]:
# your code here
response = requests.get("https://api.github.com/repos/python/cpython")
data = response.json()
print(data["stargazers_count"])

78493


---
## 4. Query parameters, and always set a timeout

The `params` dict becomes `?key=value` on the URL for you. A `timeout` stops a
dead server from hanging your program forever.

In [7]:
response = requests.get(
    "https://api.github.com/orgs/python/repos",
    params={"per_page": 3, "sort": "updated"},
    timeout=10,
)
for repo in response.json():
    print(repo["name"], repo["open_issues_count"])

cpython 9321
peps 57
mypy 3254


### Your turn
Change `per_page` to 5 and print how many repos came back.

In [8]:
# your code hereresponse = requests.get(
response = requests.get(
    "https://api.github.com/orgs/python/repos",
    params={"per_page": 5, "sort": "updated"},
    timeout=10,
)
for repo in response.json():
    print(repo["name"], repo["open_issues_count"])

cpython 9321
peps 57
mypy 3254
pymanager 23
typeshed 440


---
## 5. Handling a missing field safely

Not every response has the field you expect — an API can change shape, or you can mistype a key. `dict.get(key, default)` returns a fallback instead of raising `KeyError`; wrapping the lookup in `try`/`except KeyError` works too.

In [10]:
data = requests.get("https://api.github.com/orgs/python").json()

# data["repo_count"] would raise KeyError - that field does not exist.
# .get() returns a fallback instead of crashing:
repo_count = data.get("repo_count", "unknown")
print(repo_count)

# safe to use the same way even on a field that does exist:
print(data.get("public_repos", "unknown"))

unknown
99


### Your turn
GET `https://api.github.com/orgs/python` and print `data.get("stargazers_count", "not provided")` — that key genuinely does not exist on an org, so you should see the fallback printed, not a crash.

In [11]:
# your code here
data = requests.get("https://api.github.com/orgs/python").json()
print(data.get("stargazers_count", "not provided"))


not provided


---
## Topic 1 Challenge

GET `https://api.github.com/orgs/python/repos` with `per_page=5`, and print
each repo's name and star count, one per line, formatted with an f-string.

In [12]:
# your code here
response = requests.get(
    "https://api.github.com/orgs/python/repos",
    params={"per_page": 5, "sort": "updated"},
    timeout=10,
)
for repo in response.json():
    print(repo["name"], repo["open_issues_count"])

cpython 9322
peps 57
mypy 3254
pymanager 23
typeshed 440


---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | pulling a public dataset's metadata from an API before downloading it |
| **Cyber Security** | pulling a threat-feed or breach-notification API |
| **IT** | pulling live status from a monitoring or ticketing API |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| indexing `response` directly, e.g. `response["login"]` | `TypeError: 'Response' object is not subscriptable` | call `.json()` first |
| a key that does not exist in the JSON | `KeyError` | print the whole dict first and check the real field names; use `.get(key, default)` if the field is genuinely optional |
| no `timeout=` | the program can hang indefinitely on a dead server | always set one |
| forgetting `params=` and hand-building the URL string | works, but fragile and easy to get wrong | let `requests` build the query string |

## Go deeper

`examples/01_get_requests.py` — GET, status codes, .json(), params, timeout
`examples/02_handling_api_errors.py` — raise_for_status(), timeouts, checking status yourself, missing fields
Docs: [requests documentation](https://requests.readthedocs.io/en/latest/)

---
## Recap

| You can now | Using |
|---|---|
| ask a service for data | `requests.get(url)` |
| check it worked | `.status_code` |
| get Python data back | `.json()` |
| ask a more specific question | `params={{...}}` |
| avoid hanging forever | `timeout=10` |
| survive a field that might not be there | `data.get(key, default)` |

**Next:** Topic 2 — downloading a real file instead of calling an API.